In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from utils import extract_hand_keypoints
from logger import logging

ModuleNotFoundError: No module named 'utils'

In [ ]:
IMAGE_DATASET_DIR = "data/raw_data"
OUTPUT_CSV        = "data/isl_keypoints.csv"
SAMPLES_PER_CLASS = 300
LETTERS           = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

In [ ]:
def process_dataset():
    all_rows = []
    failed   = 0

    mp_hands = mp.solutions.hands
    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.5
    ) as hands:

        for letter in LETTERS:
            folder = os.path.join(IMAGE_DATASET_DIR, letter)
            if not os.path.exists(folder):
                print(f"⚠  Skipping {letter} — folder not found: {folder}")
                continue

            images = [f for f in os.listdir(folder)
                      if f.lower().endswith(('.jpg','.jpeg','.png'))][:SAMPLES_PER_CLASS]
            count = 0

            for img_file in tqdm(images, desc=f"Letter {letter}"):
                img = cv2.imread(os.path.join(folder, img_file))
                if img is None:
                    continue

                img = cv2.copyMakeBorder(img, 20, 20, 20, 20,
                                         cv2.BORDER_CONSTANT, value=[255,255,255])
                rgb    = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                result = hands.process(rgb)

                if result.multi_hand_landmarks:
                    kp = extract_hand_keypoints(result.multi_hand_landmarks[0])
                    all_rows.append([letter] + kp.tolist())
                    count += 1
                else:
                    failed += 1

            print(f"  ✓ {letter}: {count}/{len(images)} extracted")
            logging.info(f"{letter}: {count} samples")

    cols = ["label"] + [f"f{i}" for i in range(63)]
    df   = pd.DataFrame(all_rows, columns=cols)
    df.to_csv(OUTPUT_CSV, index=False)

    print(f"\n Saved {len(df)} rows → {OUTPUT_CSV}")
    print(f"   Missed: {failed} images (mediapipe couldn't detect hand)")
    logging.info(f"Done: {len(df)} rows, {failed} failures")

if __name__ == "__main__":
    process_dataset()

Letter A: 100%|██████████| 300/300 [00:27<00:00, 10.89it/s]


  ✓ A: 294/300 extracted


Letter B: 100%|██████████| 300/300 [00:18<00:00, 16.04it/s]


  ✓ B: 156/300 extracted


Letter C: 100%|██████████| 300/300 [00:25<00:00, 11.75it/s]


...

  ✓ X: 232/300 extracted


Letter Y: 100%|██████████| 300/300 [00:39<00:00,  7.56it/s]


  ✓ Y: 213/300 extracted


Letter Z: 100%|██████████| 300/300 [00:37<00:00,  7.93it/s]


  ✓ Z: 133/300 extracted

 Saved 5277 rows → data/isl_keypoints.csv
   Missed: 2523 images (mediapipe couldn't detect hand)
